In [24]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append('../')
import pandas as pd
import numpy as np
from src.processing import preprocess_data
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge, Lasso
import matplotlib.pyplot as plt
import xgboost as xgb
from src.visualize import plot_xgb_learning_curve, plot_feature_importance

train = pd.read_csv('../data/train.csv')
test = pd.read_csv('../data/test.csv')
outliers = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)].index
train = train.drop(outliers).reset_index(drop=True)
train_final, test_final = preprocess_data(train, test)

X = train_final.drop(['SalePrice', 'Id'], axis=1)
y = train_final['SalePrice']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [22]:
model_xgb_B = xgb.XGBRegressor(
    n_estimators=5000,
    learning_rate=0.005,   # 极低学习率
    max_depth=3,
    subsample=0.6,
    colsample_bytree=0.4,
    reg_lambda=1.5,        # 增加 L2 正则
    eval_metric="rmse",
    early_stopping_rounds=100
)

model_xgb_B.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_val, y_val)], verbose=False)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.4, device=None, early_stopping_rounds=100,
             enable_categorical=False, eval_metric='rmse', feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.005, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=3, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=5000, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [23]:
# 1. 拿到刚才实验得出的最佳迭代次数
best_n = model_xgb_B.best_iteration

# 2. 重新定义模型（去掉 early_stopping，设置 n_estimators 为 best_n）
final_model = xgb.XGBRegressor(
    n_estimators=best_n,    # 关键：不再是 5000，而是实验得出的那个数
    learning_rate=0.005,
    max_depth=3,
    subsample=0.6,
    colsample_bytree=0.4,
    reg_lambda=1.5
)

# 3. 用全量数据训练 (不再需要 eval_set)
final_model.fit(X, y)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.4, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             gamma=None, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.005, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=3, max_leaves=None,
             min_child_weight=None, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=4048, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [25]:
model_lasso = Lasso(alpha=0.0005, random_state=1)
model_lasso.fit(X, y)

D:\anaconda\envs\pytorch-env\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.900e-01, tolerance: 2.328e-02
  model = cd_fast.enet_coordinate_descent(


Lasso(alpha=0.0005, random_state=1)

In [28]:
final_model.fit(X, y)
xgb_preds = np.expm1(final_model.predict(X_test))

model_lasso.fit(X, y)
lasso_preds = np.expm1(model_lasso.predict(X_test))

final_preds = (xgb_preds * 0.7) + (lasso_preds * 0.3)

submission = pd.DataFrame({
    "Id": test_final["Id"],
    "SalePrice": final_preds
})

submission.to_csv('../submissions/blending_submission_v1.csv', index=False)
print("Blending 提交文件已生成！")

Blending 提交文件已生成！


D:\anaconda\envs\pytorch-env\lib\site-packages\sklearn\linear_model\_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 6.900e-01, tolerance: 2.328e-02
  model = cd_fast.enet_coordinate_descent(
